# Condense Paderborn for transfer to the GPU server

The raw dataset is 9.7 GB of which this project uses one channel (`vibration_1`) out of seven, at a rate it immediately downsamples. This keeps only that channel, resampled 64 kHz -> 25.6 kHz as float32, for the WHOLE dataset (every bearing code and operating condition, not just what today's config reads -- the marginal cost is small and it means no future config change forces another download at server rates).

Result is ~1 GB instead of 9.7 GB. It also removes ~72 s of `scipy.io.loadmat` from every training run, which matters once a hyperparameter sweep repeats it dozens of times.

## 1. Reconstruct the loader code

In [ ]:
import base64, pathlib
WORKDIR = pathlib.Path('/kaggle/working')
SRC_FILES = {
    'src/domain_adaptation/__init__.py': (
        'IiIiWmVyby1zaG90IGZhdWx0IGRpYWdub3NpcyB2aWEgdW5zdXBlcnZpc2VkIGRvbWFpbiBhZGFwdGF0aW9uIChEQU5OKS4KClRoaXJkLCBpbmRlcGVuZGVudCBleHBlcmltZW50IGluIHRoaXMgcmVwbyAoc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQpOgpgdmlicmF0aW9uX2FkL2AgKHNpbmdsZS1zZW5zb3IgQ1dSVSBhdXRvZW5jb2RlcikgYW5kIGBoeWRyYXVsaWNfZnVzaW9uX2FkL2AKKG11bHRpLXNlbnNvciBaZU1BIGZ1c2lvbiBhdXRvZW5jb2RlcikgYXJlIGJvdGggdW50b3VjaGVkLiBUaGlzIHBhY2thZ2UKY29tYmluZXMgQ1dSVSAoYWxyZWFkeSBsb2NhbCwgYGRhdGEvcmF3L2ApIGFuZCBQYWRlcmJvcm4gVW5pdmVyc2l0eSdzIEtBdApiZWFyaW5nIGRhdGFzZXQgKGF0dGFjaGVkIGFzIGEgS2FnZ2xlIGlucHV0IGRhdGFzZXQsIG5ldmVyIGRvd25sb2FkZWQgaW4KZnVsbCkgdG8gdHJhaW4gYSBjbGFzc2lmaWVyIHRoYXQgdHJhbnNmZXJzIGFjcm9zcyBib3RoIGRhdGFzZXRzIGFuZCBhCmhlbGQtb3V0IG9wZXJhdGluZyBjb25kaXRpb24sIHVzaW5nIG9ubHkgdGhhdCBjb25kaXRpb24ncyBIRUFMVEhZIGRhdGEgdG8KYWRhcHQgLS0gbmV2ZXIgYSBzaW5nbGUgbGFiZWxlZCAob3IgZXZlbiB1bmxhYmVsZWQpIGZhdWx0IGV4YW1wbGUgZnJvbSB0aGUKdGFyZ2V0IC0tIHRoZW4gY2hlY2tzIHdoZXRoZXIgaXQgY2FuIGRpYWdub3NlIHJlYWwgdGFyZ2V0IGZhdWx0cyBpdCBuZXZlcgp0cmFpbmVkIG9uLgoiIiIK'
    ),
    'src/domain_adaptation/data/__init__.py': (
        ''
    ),
    'src/domain_adaptation/data/pu_bearing_codes.py': (
        'IiIiQ2Fub25pY2FsIGJlYXJpbmctY29kZSAtPiBmYXVsdCBtZXRhZGF0YSBmb3IgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IC8KS0F0LURhdGFDZW50ZXIgYmVhcmluZyBkYXRhc2V0IChMZXNzbWVpZXIgZXQgYWwuIDIwMTYsICJDb25kaXRpb24gTW9uaXRvcmluZwpvZiBCZWFyaW5nIERhbWFnZSBpbiBFbGVjdHJvbWVjaGFuaWNhbCBEcml2ZSBTeXN0ZW1zIikuCgpWZXJpZmllZCBhZ2FpbnN0IHRoZSByZWFsIGRhdGFzZXQgb24gMjAyNi0wOS0xMCAoc2VlCkRPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncyAiUmV2aXNpb25zIiBzZWN0aW9uKTogZG93bmxvYWRlZCAzIHJlYWwgZmlsZXMKb2ZmIHRoZSBhdHRhY2hlZCBLYWdnbGUgbWlycm9yIChgZGlwcGF0ZWwwMy9wYWRlcmJvcm4tZGJgKSBhbmQgcGFnZWQKdGhyb3VnaCBpdHMgZnVsbCBmaWxlIGxpc3RpbmcgdmlhIHRoZSBLYWdnbGUgQVBJIHRvIGNvbmZpcm0gZXZlcnkgY29kZQpiZWxvdyBleGlzdHMsIHdpdGggMjAgcnVucyB1bmRlciBlYWNoIG9mIHRoZSBkYXRhc2V0J3MgNCBvcGVyYXRpbmcKY29uZGl0aW9ucy4KCmB1bml2ZXJzYWxfbGFiZWxgIG1hcHMgaW50byB0aGUgU0FNRSAzLWNsYXNzIHNwYWNlIENXUlUgdXNlcyAoc2VlCmN3cnVfbG9hZGVyLlVOSVZFUlNBTF9MQUJFTFMpOiAwPWhlYWx0aHksIDE9aW5uZXJfcmFjZSwgMj1vdXRlcl9yYWNlLgpDb21iaW5lZC1kYW1hZ2UgY29kZXMgKEtCKiwgYm90aCByYWNlcyBkYW1hZ2VkIG9uIG9uZSBwaHlzaWNhbCBiZWFyaW5nKQpnZXQgdW5pdmVyc2FsX2xhYmVsPU5vbmUgLS0gdGhleSBkb24ndCBmaXQgYSBzaW5nbGUtZmF1bHQtdHlwZSBsYWJlbCBhbmQKYXJlIGV4Y2x1ZGVkIGZyb20gdGhpcyBwcm9qZWN0J3MgY2xhc3NpZmljYXRpb24gdGFzazsga2VwdCBoZXJlIG9ubHkgZm9yCnJlZmVyZW5jZS9jb21wbGV0ZW5lc3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpIRUFMVEhZID0gWyJLMDAxIiwgIkswMDIiLCAiSzAwMyIsICJLMDA0IiwgIkswMDUiLCAiSzAwNiJdCkFSVElGSUNJQUxfT1VURVIgPSBbIktBMDEiLCAiS0EwMyIsICJLQTA1IiwgIktBMDYiLCAiS0EwNyIsICJLQTA4IiwgIktBMDkiXQpBUlRJRklDSUFMX0lOTkVSID0gWyJLSTAxIiwgIktJMDMiLCAiS0kwNSIsICJLSTA3IiwgIktJMDgiXQpSRUFMX09VVEVSID0gWyJLQTA0IiwgIktBMTUiLCAiS0ExNiIsICJLQTIyIiwgIktBMzAiXQpSRUFMX0lOTkVSID0gWyJLSTA0IiwgIktJMTQiLCAiS0kxNiIsICJLSTE3IiwgIktJMTgiLCAiS0kyMSJdClJFQUxfQ09NQklORUQgPSBbIktCMjMiLCAiS0IyNCIsICJLQjI3Il0KCgpkZWYgX2J1aWxkX3RhYmxlKCkgLT4gZGljdFtzdHIsIGRpY3RdOgogICAgdGFibGU6IGRpY3Rbc3RyLCBkaWN0XSA9IHt9CiAgICBmb3IgY29kZSBpbiBIRUFMVEhZOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogImhlYWx0aHkiLCAiZGFtYWdlX21vZGUiOiAibm9uZSIsICJ1bml2ZXJzYWxfbGFiZWwiOiAwfQogICAgZm9yIGNvZGUgaW4gQVJUSUZJQ0lBTF9PVVRFUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJvdXRlcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogImFydGlmaWNpYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMn0KICAgIGZvciBjb2RlIGluIEFSVElGSUNJQUxfSU5ORVI6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiaW5uZXJfcmFjZSIsICJkYW1hZ2VfbW9kZSI6ICJhcnRpZmljaWFsIiwgInVuaXZlcnNhbF9sYWJlbCI6IDF9CiAgICBmb3IgY29kZSBpbiBSRUFMX09VVEVSOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogIm91dGVyX3JhY2UiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiAyfQogICAgZm9yIGNvZGUgaW4gUkVBTF9JTk5FUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJpbm5lcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogInJlYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMX0KICAgIGZvciBjb2RlIGluIFJFQUxfQ09NQklORUQ6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiY29tYmluZWQiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiBOb25lfQogICAgcmV0dXJuIHRhYmxlCgoKUFVfQkVBUklOR19DT0RFUyA9IF9idWlsZF90YWJsZSgpCg=='
    ),
    'src/domain_adaptation/data/pu_loader.py': (
        'IiIiUGFyc2UgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IChQVSAvIEtBdC1EYXRhQ2VudGVyKSBiZWFyaW5nIGRhdGFzZXQuCgpMYXlvdXQgKGNvbmZpcm1lZCAyMDI2LTA5LTEwIGJ5IGRvd25sb2FkaW5nIDMgcmVhbCBmaWxlcyBvZmYgdGhlIGF0dGFjaGVkCkthZ2dsZSBtaXJyb3IsIGBkaXBwYXRlbDAzL3BhZGVyYm9ybi1kYmAgLS0gc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncwoiUmV2aXNpb25zIiBzZWN0aW9uKToKCiAgICA8cm9vdD4vPGJlYXJpbmdfY29kZT4vPGNvbmRpdGlvbj5fPGJlYXJpbmdfY29kZT5fPHJ1bj4ubWF0CgplLmcuIGBLMDAxL04wOV9NMDdfRjEwX0swMDFfMS5tYXRgLiBgY29uZGl0aW9uYCBlbmNvZGVzIHJvdGF0aW9uYWwgc3BlZWQKKE4wOT05MDBycG0sIE4xNT0xNTAwcnBtKSwgbG9hZCB0b3JxdWUgKE0wMT0wLjFObSwgTTA3PTAuN05tKSBhbmQgcmFkaWFsCmZvcmNlIChGMDQ9NDAwTiwgRjEwPTEwMDBOKSAtLSBzZWUgcHVfYmVhcmluZ19jb2Rlcy5weSBmb3Igd2hhdCBlYWNoCmBiZWFyaW5nX2NvZGVgIG1lYW5zIChoZWFsdGh5L2FydGlmaWNpYWwvcmVhbCwgaW5uZXIvb3V0ZXIgcmFjZSkuCgpFYWNoIGAubWF0YCBmaWxlIGlzIGEgTUFUTEFCIHN0cnVjdCAodG9wLWxldmVsIGtleSA9IHRoZSBmaWxlbmFtZSB3aXRoIHRoZQpleHRlbnNpb24gc3RyaXBwZWQpIHdpdGggYSBgWWAgZmllbGQ6IGEgbGVuZ3RoLTcgbGlzdCBvZiBwZXItY2hhbm5lbApkaWN0cywgZWFjaCBgeydOYW1lJzogc3RyLCAnRGF0YSc6IDFEIGFycmF5fWAuIENoYW5uZWxzIGFuZCB0aGVpcgphcHByb3hpbWF0ZSBuYXRpdmUgcmF0ZXMgKGZyb20gdGhlIHNhbWUgaW5zcGVjdGlvbik6IGZvcmNlL3NwZWVkL3RvcnF1ZQpAIH40a0h6LCBwaGFzZV9jdXJyZW50XzEvMiBAIH42NGtIeiwgdGVtcF8yX2JlYXJpbmdfbW9kdWxlIEAgfjEuMjVIeiAoNQpzYW1wbGVzIHBlciA0cyBydW4pLCB2aWJyYXRpb25fMSBAIH42NGtIeiAtLSBvbmx5IGB2aWJyYXRpb25fMWAgaXMgdXNlZApoZXJlLCBtYXRjaGluZyBDV1JVJ3Mgc2luZ2xlLWFjY2VsZXJvbWV0ZXItY2hhbm5lbCBzY29wZS4gUmF3IHJ1biBsZW5ndGhzCnZhcnkgYnkgYSBmZXcgaHVuZHJlZCBzYW1wbGVzIGFyb3VuZCB0aGUgbm9taW5hbCA0cy8yNTYwMDAgc2FtcGxlcyAoYSBEQVEKcXVpcmssIG5vdCBhIGJ1ZykgLS0gbm90aGluZyBkb3duc3RyZWFtIGFzc3VtZXMgYW4gZXhhY3QgbGVuZ3RoLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHNjaXB5LmlvCgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEucHVfYmVhcmluZ19jb2RlcyBpbXBvcnQgUFVfQkVBUklOR19DT0RFUwoKRklMRU5BTUVfUkUgPSByZS5jb21waWxlKHIiXig/UDxjb25kaXRpb24+TlxkezJ9X01cZHsyfV9GXGR7Mn0pXyg/UDxjb2RlPltBLVpdK1xkKylfKD9QPHJ1bj5cZCspXC5tYXQkIikKCiMgTm9taW5hbCBzaGFmdCBzcGVlZCBwZXIgUFUgb3BlcmF0aW5nLWNvbmRpdGlvbiBjb2RlLiBOZWVkZWQgZm9yIG9yZGVyCiMgdHJhY2tpbmcgKGRhdGEvb3JkZXJfdHJhY2tpbmcucHkpLCB3aGljaCBkaXZpZGVzIHNwZWVkIG91dCBvZiB0aGUKIyByZXByZXNlbnRhdGlvbi4gT25seSB0aGUgTiBmaWVsZCBzZXRzIHNoYWZ0IHJhdGUgLS0gTSAobG9hZCB0b3JxdWUpIGFuZAojIEYgKHJhZGlhbCBmb3JjZSkgZG8gbm90LgpDT05ESVRJT05fUlBNID0geyJOMTUiOiAxNTAwLCAiTjA5IjogOTAwfQoKCmRlZiBycG1fZm9yX2NvbmRpdGlvbihjb25kaXRpb246IHN0cikgLT4gaW50OgogICAgIiIiYE4wOV9NMDdfRjEwYCAtPiA5MDAuIiIiCiAgICBwcmVmaXggPSBjb25kaXRpb24uc3BsaXQoIl8iKVswXQogICAgaWYgcHJlZml4IG5vdCBpbiBDT05ESVRJT05fUlBNOgogICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBzcGVlZCBjb2RlIHtwcmVmaXghcn0gaW4gY29uZGl0aW9uIHtjb25kaXRpb24hcn0iKQogICAgcmV0dXJuIENPTkRJVElPTl9SUE1bcHJlZml4XQoKCmRlZiBmaW5kX3B1X3Jvb3QobG9jYWxfaGludDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoOgogICAgIiIiTG9jYXRlIHRoZSBQVSBkYXRhc2V0J3Mgcm9vdCBkaXJlY3RvcnkgKHRoZSBmb2xkZXIgdGhhdCBkaXJlY3RseQogICAgY29udGFpbnMgSzAwMS8sIEswMDIvLCAuLi4gc3ViZm9sZGVycykuCgogICAgT24gS2FnZ2xlLCBzZWFyY2hlcyAva2FnZ2xlL2lucHV0IGZvciBhIGZvbGRlciBjb250YWluaW5nIGEgSzAwMS8KICAgIHN1YmZvbGRlciAtLSBieSBjb250ZW50LCBub3QgYSBmaXhlZCBwYXRoLCBiZWNhdXNlIGF0dGFjaGVkLWRhdGFzZXQKICAgIG1vdW50IHBhdGhzIGhhdmUgcHJvdmVuIGluY29uc2lzdGVudCBpbiB0aGlzIHByb2plY3QgYmVmb3JlIChzZWUKICAgIHZpYnJhdGlvbl9hZCdzIG5vdGVzLzA3X2R2Y19hbmRfa2FnZ2xlLm1kKS4gRmFsbHMgYmFjayB0bwogICAgYGxvY2FsX2hpbnRgIChjb25maWdzL2RvbWFpbl9hZGFwdGF0aW9uLnlhbWwncyBwYXRocy5wdV9yb290X2xvY2FsKSBmb3IKICAgIGxvY2FsIGRldmVsb3BtZW50IGFnYWluc3QgYSBzbWFsbCBoYW5kLWJ1aWx0IHNhbXBsZS4KICAgICIiIgogICAga2FnZ2xlX2lucHV0ID0gUGF0aCgiL2thZ2dsZS9pbnB1dCIpCiAgICBpZiBrYWdnbGVfaW5wdXQuaXNfZGlyKCk6CiAgICAgICAgY2FuZGlkYXRlcyA9IHNvcnRlZChwLnBhcmVudCBmb3IgcCBpbiBrYWdnbGVfaW5wdXQucmdsb2IoIkswMDEiKSBpZiBwLmlzX2RpcigpKQogICAgICAgIGlmIGNhbmRpZGF0ZXM6CiAgICAgICAgICAgIHJldHVybiBjYW5kaWRhdGVzWzBdCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiTm8gSzAwMS8gZm9sZGVyIGZvdW5kIHVuZGVyIHtrYWdnbGVfaW5wdXR9IC0tIGlzIHRoZSBQVSBkYXRhc2V0ICIKICAgICAgICAgICAgImF0dGFjaGVkPyAoc2VlIG5vdGVib29rcy9kb21haW5fYWRhcHRhdGlvbi9rZXJuZWwtbWV0YWRhdGEuanNvbidzIGRhdGFzZXRfc291cmNlcykiCiAgICAgICAgKQogICAgaWYgbG9jYWxfaGludCBpcyBub3QgTm9uZSBhbmQgUGF0aChsb2NhbF9oaW50KS5pc19kaXIoKToKICAgICAgICByZXR1cm4gUGF0aChsb2NhbF9oaW50KQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgIk5vdCBydW5uaW5nIG9uIEthZ2dsZSAoL2thZ2dsZS9pbnB1dCBtaXNzaW5nKSBhbmQgbm8gdXNhYmxlIGxvY2FsICIKICAgICAgICBmInB1X3Jvb3QgKHtsb2NhbF9oaW50IXJ9KSAtLSBwYXNzIC0tcHUtcm9vdCBleHBsaWNpdGx5LiIKICAgICkKCgpjbGFzcyBVbnJlYWRhYmxlTWF0RmlsZShSdW50aW1lRXJyb3IpOgogICAgIiIiQSBQVSAubWF0IGZpbGUgc2NpcHkgY2Fubm90IHBhcnNlLiBSYWlzZWQgaW5zdGVhZCBvZiBsZXR0aW5nIHNjaXB5J3MKICAgIGJhcmUgYFR5cGVFcnJvcjogRXhwZWN0aW5nIG1hdHJpeCBoZXJlYCBlc2NhcGUgd2l0aCBubyBpbmRpY2F0aW9uIG9mCiAgICBXSElDSCBvZiB+MTQwMCBmaWxlcyB3YXMgYXQgZmF1bHQgLS0gd2hpY2ggaXMgZXhhY3RseSBob3cga2VybmVsIHY1CiAgICBkaWVkIGFmdGVyIDMgbWludXRlcyB3aXRoIGFuIHVuYXR0cmlidXRhYmxlIHRyYWNlYmFjay4KICAgICIiIgoKCmRlZiBsb2FkX3ZpYnJhdGlvbl9zaWduYWwobWF0X3BhdGg6IHN0ciB8IFBhdGgsIGNoYW5uZWw6IHN0ciA9ICJ2aWJyYXRpb25fMSIpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJFeHRyYWN0IG9uZSBjaGFubmVsJ3MgMUQgYXJyYXkgZnJvbSBhIFBVIC5tYXQgZmlsZS4iIiIKICAgIHRyeToKICAgICAgICBkYXRhID0gc2NpcHkuaW8ubG9hZG1hdChzdHIobWF0X3BhdGgpLCBzaW1wbGlmeV9jZWxscz1UcnVlKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIHNjaXB5IHJhaXNlcyBiYXJlIFR5cGVFcnJvci9WYWx1ZUVycm9yIHdpdGggbm8gZmlsZSBjb250ZXh0CiAgICAgICAgcmFpc2UgVW5yZWFkYWJsZU1hdEZpbGUoZiJ7bWF0X3BhdGh9OiBzY2lweSBjb3VsZCBub3QgcGFyc2UgdGhpcyAubWF0ICh7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30pIikgZnJvbSBleGMKICAgIGtleSA9IG5leHQoayBmb3IgayBpbiBkYXRhIGlmIG5vdCBrLnN0YXJ0c3dpdGgoIl9fIikpCiAgICBjaGFubmVscyA9IGRhdGFba2V5XVsiWSJdCiAgICBmb3IgY2ggaW4gY2hhbm5lbHM6CiAgICAgICAgaWYgY2hbIk5hbWUiXSA9PSBjaGFubmVsOgogICAgICAgICAgICByZXR1cm4gbnAuYXNhcnJheShjaFsiRGF0YSJdLCBkdHlwZT1ucC5mbG9hdDY0KS5yYXZlbCgpCiAgICBhdmFpbGFibGUgPSBbY2hbIk5hbWUiXSBmb3IgY2ggaW4gY2hhbm5lbHNdCiAgICByYWlzZSBLZXlFcnJvcihmInttYXRfcGF0aH06IGNoYW5uZWwge2NoYW5uZWwhcn0gbm90IGZvdW5kIChoYXM6IHthdmFpbGFibGV9KSIpCgoKZGVmIGJ1aWxkX3B1X2NhdGFsb2cocm9vdF9kaXI6IHN0ciB8IFBhdGgsIGNvZGVzOiBsaXN0W3N0cl0sIG5hdGl2ZV9mc19oejogaW50ID0gNjQwMDApIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIk9uZSByb3cgcGVyIGA8Y29kZT4vPGNvbmRpdGlvbj5fPGNvZGU+XzxydW4+Lm1hdGAgZmlsZSBmb3VuZCB1bmRlcgogICAgYHJvb3RfZGlyYCwgZm9yIGV2ZXJ5IGBjb2RlYCBpbiBgY29kZXNgIChza2lwcyB0aGUgcGVyLWNvZGUgLnBkZiBkb2NzCiAgICBhbmQgYW55dGhpbmcgdGhhdCBkb2Vzbid0IG1hdGNoIHRoZSBuYW1pbmcgY29udmVudGlvbiBvciBoYXMgbm8KICAgIHVuaXZlcnNhbC1sYWJlbCBtYXBwaW5nLCBlLmcuIGNvbWJpbmVkLWRhbWFnZSBjb2RlcyAtLSBzZWUKICAgIHB1X2JlYXJpbmdfY29kZXMucHkpLiBSb3cgb3JkZXIgaXMgZGV0ZXJtaW5pc3RpYyAoc29ydGVkIGdsb2IpLCBub3QKICAgIGRpc2sgb3JkZXIuCgogICAgYG5hdGl2ZV9mc19oemAgaXMgYSBwYXJhbWV0ZXIgKG5vdCBoYXJkY29kZWQpIHNvIHRlc3RzIGNhbiBleGVyY2lzZQogICAgdGhpcyBhZ2FpbnN0IHNtYWxsIHN5bnRoZXRpYyBzaWduYWxzIGF0IGEgbG93ZXIgcmF0ZSAtLSBwcm9kdWN0aW9uCiAgICBjYWxsZXJzIHBhc3MgY29uZmlnLnB1Lm5hdGl2ZV9mc19oeiAoNjQwMDAgZm9yIHRoZSByZWFsIGRhdGFzZXQpLgoKICAgIFJldHVybnMgY29sdW1uczogZmlsZXBhdGgsIGNvZGUsIGNvbmRpdGlvbiwgcnVuLCBycG0sIGZhdWx0X3R5cGUsCiAgICBkYW1hZ2VfbW9kZSwgdW5pdmVyc2FsX2xhYmVsLCBmc19uYXRpdmVfaHosIGRvbWFpbi4KICAgICIiIgogICAgcm9vdF9kaXIgPSBQYXRoKHJvb3RfZGlyKQogICAgcm93cyA9IFtdCiAgICBmb3IgY29kZSBpbiBjb2RlczoKICAgICAgICBjb2RlX2RpciA9IHJvb3RfZGlyIC8gY29kZQogICAgICAgIGlmIG5vdCBjb2RlX2Rpci5pc19kaXIoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJQVSBiZWFyaW5nIGNvZGUgZm9sZGVyIG5vdCBmb3VuZDoge2NvZGVfZGlyfSIpCiAgICAgICAgbWV0YSA9IFBVX0JFQVJJTkdfQ09ERVMuZ2V0KGNvZGUpCiAgICAgICAgaWYgbWV0YSBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gUFUgYmVhcmluZyBjb2RlIHtjb2RlIXJ9IC0tIG5vdCBpbiBwdV9iZWFyaW5nX2NvZGVzLlBVX0JFQVJJTkdfQ09ERVMiKQogICAgICAgIGlmIG1ldGFbInVuaXZlcnNhbF9sYWJlbCJdIGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlICAjIGNvbWJpbmVkLWRhbWFnZSBjb2RlLCBleGNsdWRlZCBmcm9tIHRoaXMgdGFzayAoc2VlIG1vZHVsZSBkb2NzdHJpbmcpCgogICAgICAgIGZvciBtYXRfcGF0aCBpbiBzb3J0ZWQoY29kZV9kaXIuZ2xvYigiKi5tYXQiKSk6CiAgICAgICAgICAgIG0gPSBGSUxFTkFNRV9SRS5tYXRjaChtYXRfcGF0aC5uYW1lKQogICAgICAgICAgICBpZiBub3QgbSBvciBtLmdyb3VwKCJjb2RlIikgIT0gY29kZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJmaWxlcGF0aCI6IHN0cihtYXRfcGF0aCksCiAgICAgICAgICAgICAgICAgICAgImNvZGUiOiBjb2RlLAogICAgICAgICAgICAgICAgICAgICJjb25kaXRpb24iOiBtLmdyb3VwKCJjb25kaXRpb24iKSwKICAgICAgICAgICAgICAgICAgICAicnVuIjogaW50KG0uZ3JvdXAoInJ1biIpKSwKICAgICAgICAgICAgICAgICAgICAicnBtIjogcnBtX2Zvcl9jb25kaXRpb24obS5ncm91cCgiY29uZGl0aW9uIikpLAogICAgICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogbWV0YVsiZmF1bHRfdHlwZSJdLAogICAgICAgICAgICAgICAgICAgICJkYW1hZ2VfbW9kZSI6IG1ldGFbImRhbWFnZV9tb2RlIl0sCiAgICAgICAgICAgICAgICAgICAgInVuaXZlcnNhbF9sYWJlbCI6IG1ldGFbInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJmc19uYXRpdmVfaHoiOiBuYXRpdmVfZnNfaHosCiAgICAgICAgICAgICAgICAgICAgImRvbWFpbiI6ICJwdSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIGlmIG5vdCByb3dzOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTm8gbWF0Y2hpbmcgUFUgLm1hdCBmaWxlcyBmb3VuZCB1bmRlciB7cm9vdF9kaXJ9IGZvciBjb2RlcyB7Y29kZXN9IikKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykKCiMgLS0tIGNvbmRlbnNlZCAoInByZXBhcmVkIikgc3RvcmUgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgcHJlcGFyZV9wdS5weSB0dXJucyB0aGUgOS43IEdCIHJhdyBkYXRhc2V0IGludG8gfjEgR0Igb2YgcGVyLWNvZGUgLm5wegojIGFyY2hpdmVzLiBFdmVyeXRoaW5nIGJlbG93IHJlYWRzIHRoYXQgZm9ybS4gRGV0ZWN0aW9uIGlzIGF1dG9tYXRpYywgc28gdGhlCiMgc2FtZSAtLXB1LXJvb3Qgd29ya3Mgd2hldGhlciBpdCBwb2ludHMgYXQgcmF3IC5tYXQgZm9sZGVycyBvciBhIGNvbmRlbnNlZAojIHN0b3JlIC0tIG5vIGNvbmZpZyBmbGFnIHRvIGZvcmdldCB0byBmbGlwLCBhbmQgbm8gc2Vjb25kIGNvZGUgcGF0aCB0aHJvdWdoCiMgdGhlIHJlc3Qgb2YgdGhlIHBpcGVsaW5lLgoKCmRlZiBpc19wcmVwYXJlZF9zdG9yZShyb290OiBzdHIgfCBQYXRoKSAtPiBib29sOgogICAgIiIiQSBjb25kZW5zZWQgc3RvcmUgaXMgaWRlbnRpZmllZCBieSBpdHMgbWFuaWZlc3Q7IHJhdyBQVSBoYXMgbm9uZS4iIiIKICAgIHJldHVybiAoUGF0aChyb290KSAvICJtYW5pZmVzdC5jc3YiKS5pc19maWxlKCkKCgpjbGFzcyBQcmVwYXJlZFBVU3RvcmU6CiAgICAiIiJSZWFkcyB0aGUgcGVyLWNvZGUgYXJjaGl2ZXMgd3JpdHRlbiBieSBwcmVwYXJlX3B1LnB5LgoKICAgIG51bXB5IGxvYWRzIC5ucHogbWVtYmVycyBsYXppbHksIHNvIGhvbGRpbmcgb25lIG9wZW4gaGFuZGxlIHBlciBiZWFyaW5nCiAgICBjb2RlIGFuZCBpbmRleGluZyBpbnRvIGl0IGF2b2lkcyByZS1yZWFkaW5nIGEgfjMzIE1CIGFyY2hpdmUgZm9yIGV2ZXJ5CiAgICBvbmUgb2YgaXRzIDgwIHJ1bnMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcm9vdDogc3RyIHwgUGF0aCk6CiAgICAgICAgc2VsZi5yb290ID0gUGF0aChyb290KQogICAgICAgIG1hbmlmZXN0X3BhdGggPSBzZWxmLnJvb3QgLyAibWFuaWZlc3QuY3N2IgogICAgICAgIGlmIG5vdCBtYW5pZmVzdF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7c2VsZi5yb290fSBpcyBub3QgYSBwcmVwYXJlZCBzdG9yZSAobm8gbWFuaWZlc3QuY3N2KSIpCiAgICAgICAgc2VsZi5tYW5pZmVzdCA9IHBkLnJlYWRfY3N2KG1hbmlmZXN0X3BhdGgpCiAgICAgICAgc2VsZi5faGFuZGxlczogZGljdFtzdHIsIG9iamVjdF0gPSB7fQoKICAgIGRlZiBfaGFuZGxlKHNlbGYsIGNvZGU6IHN0cik6CiAgICAgICAgaWYgY29kZSBub3QgaW4gc2VsZi5faGFuZGxlczoKICAgICAgICAgICAgcGF0aCA9IHNlbGYucm9vdCAvIGYie2NvZGV9Lm5weiIKICAgICAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7cGF0aH0gbWlzc2luZyAtLSBtYW5pZmVzdCBsaXN0cyB7Y29kZX0gYnV0IHRoZSBhcmNoaXZlIGlzIGFic2VudCIpCiAgICAgICAgICAgIHNlbGYuX2hhbmRsZXNbY29kZV0gPSBucC5sb2FkKHBhdGgpCiAgICAgICAgcmV0dXJuIHNlbGYuX2hhbmRsZXNbY29kZV0KCiAgICBkZWYgbG9hZChzZWxmLCBjb2RlOiBzdHIsIGNvbmRpdGlvbjogc3RyLCBydW46IGludCkgLT4gbnAubmRhcnJheToKICAgICAgICBrZXkgPSBmIntjb25kaXRpb259X3tydW59IgogICAgICAgIGhhbmRsZSA9IHNlbGYuX2hhbmRsZShjb2RlKQogICAgICAgIGlmIGtleSBub3QgaW4gaGFuZGxlOgogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIntjb2RlfS5ucHogaGFzIG5vIHJ1biB7a2V5IXJ9IikKICAgICAgICByZXR1cm4gbnAuYXNhcnJheShoYW5kbGVba2V5XSwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICBkZWYgY2F0YWxvZyhzZWxmLCBjb2RlczogbGlzdFtzdHJdKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiU2FtZSBjb2x1bW5zIGJ1aWxkX3B1X2NhdGFsb2cgcmV0dXJucywgc28gZG93bnN0cmVhbSBjb2RlIGlzCiAgICAgICAgaWRlbnRpY2FsIGZvciBlaXRoZXIgc3RvcmFnZSBmb3JtLiIiIgogICAgICAgIHdhbnRlZCA9IHNlbGYubWFuaWZlc3Rbc2VsZi5tYW5pZmVzdC5jb2RlLmlzaW4oY29kZXMpXS5jb3B5KCkKICAgICAgICBtaXNzaW5nID0gc29ydGVkKHNldChjb2RlcykgLSBzZXQod2FudGVkLmNvZGUudW5pcXVlKCkpKQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUHJlcGFyZWQgc3RvcmUge3NlbGYucm9vdH0gaGFzIG5vIGRhdGEgZm9yIGNvZGVzIHttaXNzaW5nfSIpCiAgICAgICAgd2FudGVkID0gd2FudGVkW3dhbnRlZC51bml2ZXJzYWxfbGFiZWwubm90bmEoKV0KICAgICAgICB3YW50ZWRbInVuaXZlcnNhbF9sYWJlbCJdID0gd2FudGVkLnVuaXZlcnNhbF9sYWJlbC5hc3R5cGUoaW50KQogICAgICAgIHdhbnRlZFsiZmlsZXBhdGgiXSA9ICIiICAgICAgICAgICMgc2lnbmFscyBsaXZlIGluc2lkZSB0aGUgYXJjaGl2ZXMsIG5vdCBhcyBmaWxlcwogICAgICAgIHdhbnRlZFsiZnNfbmF0aXZlX2h6Il0gPSB3YW50ZWRbImZzX2h6Il0KICAgICAgICB3YW50ZWRbImRvbWFpbiJdID0gInB1IgogICAgICAgIGNvbHMgPSBbImZpbGVwYXRoIiwgImNvZGUiLCAiY29uZGl0aW9uIiwgInJ1biIsICJycG0iLCAiZmF1bHRfdHlwZSIsCiAgICAgICAgICAgICAgICAiZGFtYWdlX21vZGUiLCAidW5pdmVyc2FsX2xhYmVsIiwgImZzX25hdGl2ZV9oeiIsICJkb21haW4iXQogICAgICAgIHJldHVybiB3YW50ZWRbY29sc10ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQo='
    ),
    'src/domain_adaptation/data/transforms.py': (
        'IiIiQ3Jvc3MtZG9tYWluIHNhbXBsZS1yYXRlIGFsaWdubWVudC4KCkNXUlUncyBEcml2ZS1FbmQgc2lnbmFsIGlzIGFscmVhZHkgMTJrSHo7IFBhZGVyYm9ybidzIHZpYnJhdGlvbl8xIGNoYW5uZWwKaXMgNjRrSHouIEEgQ05OIHRyYWluZWQgb24gbG9nLVNURlQtbWFnbml0dWRlIGltYWdlcyBuZWVkcyBwaXhlbHMgdGhhdAptZWFuIHRoZSBzYW1lIHBoeXNpY2FsIGZyZXF1ZW5jeSByYW5nZSBpbiBldmVyeSBkb21haW4sIHNvIGV2ZXJ5IHNpZ25hbCBpcwpyZXNhbXBsZWQgdG8gb25lIGNvbW1vbiByYXRlIChgY29uZmlnLmNvbW1vbi5mc19oemAsIGRlZmF1bHQgMTIwMDApIEJFRk9SRQp3aW5kb3dpbmcvU1RGVCAtLSBzZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiIHNlY3Rpb24gZm9yCndoeSAxMmtIeiAoQ1dSVSdzIG5hdGl2ZSByYXRlKSB3YXMgY2hvc2VuIG92ZXIgdXAgc2FtcGxpbmcgZXZlcnl0aGluZyB0bwo0OGtIeiBhcyB0aGUgb3JpZ2luYWwgcGxhbiBwcm9wb3NlZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZnJhY3Rpb25zIGltcG9ydCBGcmFjdGlvbgoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuc2lnbmFsIGltcG9ydCByZXNhbXBsZV9wb2x5CgoKZGVmIHJlc2FtcGxlX3RvX3JhdGUoc2lnbmFsOiBucC5uZGFycmF5LCBuYXRpdmVfZnNfaHo6IGludCwgdGFyZ2V0X2ZzX2h6OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQb2x5cGhhc2UgcmVzYW1wbGUgKHByb3BlciBhbnRpLWFsaWFzaW5nIGZpbHRlciwgdW5saWtlIGEgbmFpdmUKICAgIEZGVC1iaW4gcmVzYW1wbGUpIGZyb20gbmF0aXZlX2ZzX2h6IHRvIHRhcmdldF9mc19oei4KCiAgICBOby1vcHMgKHJldHVybnMgYHNpZ25hbGAgdW5jaGFuZ2VkLCBub3QgZXZlbiBjb3BpZWQpIHdoZW4gdGhlIHJhdGVzCiAgICBhbHJlYWR5IG1hdGNoIC0tIGUuZy4gQ1dSVSBhdCAxMmtIeiB3aGVuIHRhcmdldF9mc19oej0xMjAwMCAtLSBzbyBkYXRhCiAgICB0aGF0IGRvZXNuJ3QgbmVlZCByZXNhbXBsaW5nIG5ldmVyIHBpY2tzIHVwIGZpbHRlciBhcnRpZmFjdHMgZnJvbSBvbmUuCiAgICAiIiIKICAgIGlmIG5hdGl2ZV9mc19oeiA9PSB0YXJnZXRfZnNfaHo6CiAgICAgICAgcmV0dXJuIHNpZ25hbAogICAgcmF0aW8gPSBGcmFjdGlvbih0YXJnZXRfZnNfaHosIG5hdGl2ZV9mc19oeikubGltaXRfZGVub21pbmF0b3IoMTAwMCkKICAgIHJldHVybiByZXNhbXBsZV9wb2x5KHNpZ25hbCwgdXA9cmF0aW8ubnVtZXJhdG9yLCBkb3duPXJhdGlvLmRlbm9taW5hdG9yKS5hc3R5cGUoc2lnbmFsLmR0eXBlKQo='
    ),
    'src/domain_adaptation/data/prepare_pu.py': (
        'IiIiQ29uZGVuc2UgdGhlIFBhZGVyYm9ybiBkYXRhc2V0IGludG8gYSBjb21wYWN0LCB0cmFuc2Zlci1mcmllbmRseSBmb3JtLgoKUnVucyBPTiBLQUdHTEUsIHdoZXJlIHRoZSA5LjcgR0IgZGF0YXNldCBpcyBhbHJlYWR5IG1vdW50ZWQgYW5kIGNvbXB1dGUgaXMKZnJlZS4gV3JpdGVzIG9uZSBgLm5wemAgcGVyIGJlYXJpbmcgY29kZSBob2xkaW5nIGV2ZXJ5IHJ1bidzIHZpYnJhdGlvbgpzaWduYWwsIHJlc2FtcGxlZCBhbmQgc3RvcmVkIGFzIGZsb2F0MzIuIFRoZSByZXN1bHQgaXMgfjEgR0IgZm9yIHRoZSBlbnRpcmUKZGF0YXNldCwgd2hpY2ggaXMgd2hhdCBhY3R1YWxseSBnZXRzIG1vdmVkIHRvIHRoZSBHUFUgc2VydmVyLgoKV2h5IHRoaXMgZXhpc3RzIHJhdGhlciB0aGFuIGNvcHlpbmcgdGhlIHJhdyBkYXRhc2V0OgoKKiAqKlNldmVuIGNoYW5uZWxzIGluLCBvbmUgY2hhbm5lbCBvdXQuKiogRWFjaCBgLm1hdGAgaXMgfjguNyBNQiBjYXJyeWluZwogIGZvcmNlLCB0d28gcGhhc2UgY3VycmVudHMsIHNwZWVkLCB0ZW1wZXJhdHVyZSwgdG9ycXVlIGFuZCB2aWJyYXRpb24uIFRoaXMKICBwcm9qZWN0IHVzZXMgYHZpYnJhdGlvbl8xYCBhbG9uZSAtLSBldmVyeXRoaW5nIGVsc2UgaXMgYnl0ZXMgd2Ugd291bGQKICBwYXkgdG8gdHJhbnNmZXIsIHN0b3JlLCBhbmQgdGhlbiBuZXZlciByZWFkLgoqICoqNjQga0h6IGluLCAxMiBrSHogb3V0LioqIFRoZSBwaXBlbGluZSByZXNhbXBsZXMgUFUgZG93biBiZWZvcmUgZG9pbmcKICBhbnl0aGluZyB3aXRoIGl0LCBzbyBzaGlwcGluZyB0aGUgb3JpZ2luYWwgcmF0ZSBtZWFucyBwYXlpbmcgdG8gbW92ZQogIHNhbXBsZXMgdGhhdCBnZXQgZGlzY2FyZGVkIG9uIGFycml2YWwuCiogKipgc2NpcHkuaW8ubG9hZG1hdGAgaXMgc2xvdy4qKiBCdWlsZGluZyB0aGUgZGF0YXNldCBmcm9tIDE0MjAgcmF3IGAubWF0YAogIGZpbGVzIGNvc3QgfjcyIHMgb2YgZXZlcnkgS2FnZ2xlIHJ1bi4gUmVhZGluZyBwcmUtZXh0cmFjdGVkIGFycmF5cyBpcyBhCiAgZmFzdCBgbnAubG9hZGAsIHdoaWNoIG1hdHRlcnMgb25jZSBhIGh5cGVycGFyYW1ldGVyIHN3ZWVwIHJ1bnMgaXQgZG96ZW5zCiAgb2YgdGltZXMuCiogKipSZW50ZWQgY29tcHV0ZSBpcyBiaWxsZWQgYnkgdGhlIG1pbnV0ZS4qKiBBIDEgR0IgdHJhbnNmZXIgaW5zdGVhZCBvZiBhCiAgOS43IEdCIG9uZSBpcyB0aGUgZGlmZmVyZW5jZSBwZXIgc2VydmVyIHJlYnVpbGQuCgpEZWxpYmVyYXRlbHkgY29udmVydHMgdGhlIFdIT0xFIGRhdGFzZXQsIG5vdCBqdXN0IHRoZSBzdWJzZXQgdGhlIGN1cnJlbnQKY29uZmlnIHJlYWRzLiBUaGUgbWFyZ2luYWwgY29zdCBpcyBzbWFsbCAofjEgR0IgdnMgfjAuNyBHQikgYW5kIGl0IHJlbW92ZXMKYW55IGNoYW5jZSBvZiBhIGZ1dHVyZSBjb25maWcgY2hhbmdlIC0tIGEgZm91cnRoIHNvdXJjZSBjb25kaXRpb24sIG1vcmUKYmVhcmluZyBjb2RlcywgdGhlIGNvbWJpbmVkLWRhbWFnZSBLQiogdW5pdHMgLS0gZm9yY2luZyBhbm90aGVyIGRvd25sb2FkIGF0CnNlcnZlciByYXRlcy4KCjI1LjYga0h6IGlzIGEgY2xlYW4gMi81IHN0ZXAgZG93biBmcm9tIDY0IGtIeiBhbmQgc2l0cyBhYm92ZSB0aGUgMTIga0h6IHRoZQpjcm9zcy1kYXRhc2V0IHBpcGVsaW5lIGN1cnJlbnRseSB1c2VzIChDV1JVJ3MgREUxMiBmaWxlcyBjYXAgdGhlICpjb21tb24qCnJhdGUgYXQgMTIga0h6IHJlZ2FyZGxlc3MpLCBsZWF2aW5nIGhlYWRyb29tIGZvciBQVS1vbmx5IGFuZCBvcmRlci1kb21haW4KZXhwZXJpbWVudHMgd2l0aG91dCBhIHNlY29uZCB0cmlwLgoKVXNhZ2UgKG9uIEthZ2dsZSk6CiAgICBweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wcmVwYXJlX3B1IC0tcHUtcm9vdCA8bW91bnQ+IC0tb3V0IHByZXBhcmVkX3B1CiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wdV9iZWFyaW5nX2NvZGVzIGltcG9ydCBQVV9CRUFSSU5HX0NPREVTCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wdV9sb2FkZXIgaW1wb3J0ICgKICAgIEZJTEVOQU1FX1JFLAogICAgVW5yZWFkYWJsZU1hdEZpbGUsCiAgICBsb2FkX3ZpYnJhdGlvbl9zaWduYWwsCiAgICBycG1fZm9yX2NvbmRpdGlvbiwKKQpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEudHJhbnNmb3JtcyBpbXBvcnQgcmVzYW1wbGVfdG9fcmF0ZQoKTkFUSVZFX0ZTX0haID0gNjQwMDAKUFJFUEFSRURfRlNfSFogPSAxMjAwMCAgIyA2NDAwMCAqIDMvMTYgLS0gZXhhY3QgcmF0aW9uYWwgc3RlcAojIDEyIGtIeiByYXRoZXIgdGhhbiBzb21ldGhpbmcgaGlnaGVyIGJlY2F1c2UgaXQgaXMgd2hhdCB0aGUgcGlwZWxpbmUKIyByZXNhbXBsZXMgdG8gYW55d2F5LCBhbmQgQ1dSVSdzIERFMTIgZmlsZXMgY2FwIHRoZSBjcm9zcy1kYXRhc2V0IGNvbW1vbgojIHJhdGUgdGhlcmUgcmVnYXJkbGVzcyAtLSBzbyB0aGlzIGlzIGxvc3NsZXNzIGZvciBvdXIgdXNlIHdoaWxlIGhhbHZpbmcKIyB0aGUgdHJhbnNmZXIgKG1lYXN1cmVkOiB6ZXJvIGVycm9yIGluIHRoZSBub3JtYWxpemVkIHNwZWN0cm9ncmFtIHRoZQojIG1vZGVsIGNvbnN1bWVzKS4gSXQgc3RpbGwgc3VwcG9ydHMgMjU2IHNhbXBsZXMvcmV2IGZvciBvcmRlciB0cmFja2luZwojIChOeXF1aXN0IG9yZGVyIDEyOCA9IDMuMiBrSHogYXQgMTUwMCBycG0sIGNvbWZvcnRhYmx5IGluc2lkZSA2IGtIeikuCiMgaW50MTYgd291bGQgaGFsdmUgaXQgYWdhaW4gYnV0IHBlcnR1cmJzIDIuMjklIG9mIGlucHV0IHBpeGVscyBmb3IgfjE1IHMKIyBvZiB0cmFuc2ZlciAtLSBtZWFzdXJlZCBhbmQgcmVqZWN0ZWQuIFJlZ2VuZXJhdGluZyBhdCBhbnkgb3RoZXIKIyByYXRlL3ByZWNpc2lvbiBpcyBhIGZyZWUgfjYtbWludXRlIEthZ2dsZSBydW4uCmRlZiBwcmVwYXJlX2NvZGUoY29kZV9kaXI6IFBhdGgsIG91dF9kaXI6IFBhdGgsIGZzX291dDogaW50ID0gUFJFUEFSRURfRlNfSFopIC0+IGRpY3Q6CiAgICAiIiJDb25kZW5zZSBldmVyeSBydW4gb2Ygb25lIGJlYXJpbmcgY29kZSBpbnRvIGEgc2luZ2xlIC5ucHouCgogICAgQXJyYXlzIGFyZSBrZXllZCBgPGNvbmRpdGlvbj5fPHJ1bj5gOyBhbiB1bnJlYWRhYmxlIHNvdXJjZSBmaWxlIGlzCiAgICBza2lwcGVkIGFuZCBjb3VudGVkIHJhdGhlciB0aGFuIGFib3J0aW5nLCB0aGUgc2FtZSBwb3N0dXJlCiAgICBkYXRhL2RhdGFzZXQucHkgdGFrZXMgKG9uZSBjb3JydXB0IGZpbGUgaW4gdGhpcyBLYWdnbGUgbWlycm9yLAogICAgS0EwOC9OMTVfTTAxX0YxMF9LQTA4XzIubWF0LCBvbmNlIGtpbGxlZCBhbiBlbnRpcmUgcnVuKS4KICAgICIiIgogICAgY29kZSA9IGNvZGVfZGlyLm5hbWUKICAgIGFycmF5cywgcm93cywgdW5yZWFkYWJsZSA9IHt9LCBbXSwgW10KICAgIGZvciBtYXRfcGF0aCBpbiBzb3J0ZWQoY29kZV9kaXIuZ2xvYigiKi5tYXQiKSk6CiAgICAgICAgbSA9IEZJTEVOQU1FX1JFLm1hdGNoKG1hdF9wYXRoLm5hbWUpCiAgICAgICAgaWYgbm90IG0gb3IgbS5ncm91cCgiY29kZSIpICE9IGNvZGU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY29uZGl0aW9uLCBydW4gPSBtLmdyb3VwKCJjb25kaXRpb24iKSwgaW50KG0uZ3JvdXAoInJ1biIpKQogICAgICAgIHRyeToKICAgICAgICAgICAgc2lnbmFsID0gbG9hZF92aWJyYXRpb25fc2lnbmFsKG1hdF9wYXRoKQogICAgICAgIGV4Y2VwdCBVbnJlYWRhYmxlTWF0RmlsZSBhcyBleGM6CiAgICAgICAgICAgIHVucmVhZGFibGUuYXBwZW5kKG1hdF9wYXRoLm5hbWUpCiAgICAgICAgICAgIHByaW50KGYiICAgIFNLSVAgdW5yZWFkYWJsZToge2V4Y30iKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlc2FtcGxlZCA9IHJlc2FtcGxlX3RvX3JhdGUoc2lnbmFsLCBOQVRJVkVfRlNfSFosIGZzX291dCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgYXJyYXlzW2Yie2NvbmRpdGlvbn1fe3J1bn0iXSA9IHJlc2FtcGxlZAogICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiY29kZSI6IGNvZGUsICJjb25kaXRpb24iOiBjb25kaXRpb24sICJydW4iOiBydW4sCiAgICAgICAgICAgICAgICAicnBtIjogcnBtX2Zvcl9jb25kaXRpb24oY29uZGl0aW9uKSwgImZzX2h6IjogZnNfb3V0LAogICAgICAgICAgICAgICAgIm5fc2FtcGxlcyI6IGxlbihyZXNhbXBsZWQpLAogICAgICAgICAgICAgICAgImZhdWx0X3R5cGUiOiBQVV9CRUFSSU5HX0NPREVTW2NvZGVdWyJmYXVsdF90eXBlIl0sCiAgICAgICAgICAgICAgICAiZGFtYWdlX21vZGUiOiBQVV9CRUFSSU5HX0NPREVTW2NvZGVdWyJkYW1hZ2VfbW9kZSJdLAogICAgICAgICAgICAgICAgInVuaXZlcnNhbF9sYWJlbCI6IFBVX0JFQVJJTkdfQ09ERVNbY29kZV1bInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgb3V0X3BhdGggPSBvdXRfZGlyIC8gZiJ7Y29kZX0ubnB6IgogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChvdXRfcGF0aCwgKiphcnJheXMpCiAgICBtYiA9IG91dF9wYXRoLnN0YXQoKS5zdF9zaXplIC8gMTAyNCoqMgogICAgcHJpbnQoZiIgIHtjb2RlfToge2xlbihhcnJheXMpfSBydW5zIC0+IHtvdXRfcGF0aC5uYW1lfSAoe21iOi4xZn0gTUIpIgogICAgICAgICAgKyAoZiIgIFt7bGVuKHVucmVhZGFibGUpfSB1bnJlYWRhYmxlXSIgaWYgdW5yZWFkYWJsZSBlbHNlICIiKSkKICAgIHJldHVybiB7InJvd3MiOiByb3dzLCAidW5yZWFkYWJsZSI6IHVucmVhZGFibGUsICJtYiI6IG1ifQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXB1LXJvb3QiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJNb3VudGVkIFBhZGVyYm9ybiBkYXRhc2V0IHJvb3QuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iRGlyZWN0b3J5IGZvciB0aGUgY29uZGVuc2VkIC5ucHogZmlsZXMuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZnMiLCB0eXBlPWludCwgZGVmYXVsdD1QUkVQQVJFRF9GU19IWikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29kZXMiLCBuYXJncz0iKiIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iQmVhcmluZyBjb2RlcyB0byBjb252ZXJ0IChkZWZhdWx0OiBldmVyeSBjb2RlIGZvdW5kKS4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICByb290LCBvdXRfZGlyID0gUGF0aChhcmdzLnB1X3Jvb3QpLCBQYXRoKGFyZ3Mub3V0KQogICAgY29kZXMgPSBhcmdzLmNvZGVzIG9yIHNvcnRlZCgKICAgICAgICBkLm5hbWUgZm9yIGQgaW4gcm9vdC5pdGVyZGlyKCkgaWYgZC5pc19kaXIoKSBhbmQgcmUuZnVsbG1hdGNoKHIiS1tBQkldP1xkKyIsIGQubmFtZSkKICAgICkKICAgIHByaW50KGYiQ29uZGVuc2luZyB7bGVuKGNvZGVzKX0gYmVhcmluZyBjb2RlcyBmcm9tIHtyb290fSBhdCB7YXJncy5mc30gSHpcbiIpCgogICAgYWxsX3Jvd3MsIGFsbF91bnJlYWRhYmxlLCB0b3RhbF9tYiA9IFtdLCBbXSwgMC4wCiAgICBmb3IgY29kZSBpbiBjb2RlczoKICAgICAgICByZXN1bHQgPSBwcmVwYXJlX2NvZGUocm9vdCAvIGNvZGUsIG91dF9kaXIsIGFyZ3MuZnMpCiAgICAgICAgYWxsX3Jvd3MgKz0gcmVzdWx0WyJyb3dzIl0KICAgICAgICBhbGxfdW5yZWFkYWJsZSArPSByZXN1bHRbInVucmVhZGFibGUiXQogICAgICAgIHRvdGFsX21iICs9IHJlc3VsdFsibWIiXQoKICAgIG1hbmlmZXN0ID0gcGQuRGF0YUZyYW1lKGFsbF9yb3dzKQogICAgbWFuaWZlc3QudG9fY3N2KG91dF9kaXIgLyAibWFuaWZlc3QuY3N2IiwgaW5kZXg9RmFsc2UpCiAgICBwcmludChmIlxue2xlbihtYW5pZmVzdCl9IHJ1bnMgYWNyb3NzIHttYW5pZmVzdC5jb2RlLm51bmlxdWUoKX0gY29kZXMsIHt0b3RhbF9tYjouMGZ9IE1CIHRvdGFsIikKICAgIHByaW50KGYibWFuaWZlc3QgLT4ge291dF9kaXIgLyAnbWFuaWZlc3QuY3N2J30iKQogICAgaWYgYWxsX3VucmVhZGFibGU6CiAgICAgICAgcHJpbnQoZiJ7bGVuKGFsbF91bnJlYWRhYmxlKX0gdW5yZWFkYWJsZSBzb3VyY2UgZmlsZXMgc2tpcHBlZDoge2FsbF91bnJlYWRhYmxlWzo1XX0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
    ),
}
for relpath, b64 in SRC_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(SRC_FILES)} source files')

## 2. Locate the attached dataset

In [ ]:
import sys, os
sys.path.insert(0, str(WORKDIR / 'src'))
os.environ['PYTHONPATH'] = str(WORKDIR / 'src')
os.chdir(WORKDIR)
from domain_adaptation.data.pu_loader import find_pu_root
PU_ROOT = find_pu_root()
codes = sorted(d.name for d in PU_ROOT.iterdir() if d.is_dir())
print(f'PU_ROOT = {PU_ROOT}')
print(f'{len(codes)} bearing codes: {codes}')

## 3. Condense
Every code, every condition, every run. Unreadable source files are named and skipped rather than aborting -- this mirror has at least one (`KA08/N15_M01_F10_KA08_2.mat`), and it previously killed a whole run.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'domain_adaptation.data.prepare_pu',
                '--pu-root', str(PU_ROOT), '--out', 'prepared_pu'], check=True)

## 4. Confirm what will be transferred

In [ ]:
import pandas as pd
out = WORKDIR / 'prepared_pu'
files = sorted(out.glob('*.npz'))
total_mb = sum(f.stat().st_size for f in files) / 1024**2
manifest = pd.read_csv(out / 'manifest.csv')
print(f'{len(files)} archives, {total_mb:.0f} MB total '
      f'(raw dataset is ~9,700 MB -> {9700/max(total_mb,1):.0f}x smaller)')
print(f'{len(manifest)} runs, {manifest.code.nunique()} codes, '
      f'{manifest.condition.nunique()} conditions, {manifest.fs_hz.iloc[0]} Hz')
print()
print(manifest.groupby(['damage_mode','fault_type']).size().to_string())